In [ ]:
import sys
import healpy as hp
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Manipulating one realization

In [ ]:
data_path = "/users/stevensonb/scratch/mlpng-fits/"
base1 = "alm_l_0001_v3.fits"
base2 = "alm_nl_0001_v3.fits"

with fits.open(data_path + base1) as hdul:
#hdul = fits.open(data_path + base1)
    print(hdul[1].header) # Printing more information for data check

alm_l = hp.read_alm(data_path + base1, hdu=(1, 2, 3))
alm_nl = hp.read_alm(data_path + base2, hdu=(1, 2, 3))

In [ ]:
f_NL = 100#250
alm = alm_l + (f_NL * alm_nl)
ngmap = hp.alm2map(alm, 1024, pol=True)

In [ ]:
hp.projview(
    ngmap[0],
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    max=0.0002,
    min=-0.0002,
    unit="$\mu$K",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="horizontal",
    projection_type="mollweide",
);

In [ ]:
# cartview, with labels and graticule
hp.projview(
    ngmap[0],
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="$\mu$K",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="horizontal",
    projection_type="cart",
    title="Cart projection",
);

In [ ]:
# This info is no longer needed
#10 x 10 degree maps
# res_arcmin = 2.34375
# xnpix = ynpix = 256
# m_deg_size = int(xnpix * (res_arcmin / 60))
# print(f"Maps are {m_deg_size}x{m_deg_size} deg")

hp.cartview(ngmap[0], xsize=256, ysize=256, rot=[0, 0], lonra=[0, 60], latra=[0, 10], title="CartView", unit="mK", format="%.2g", return_projected_map=False)
test_map = hp.cartview(ngmap[0], xsize=256, ysize=256, rot=[0, 0], lonra=[0, 10], latra=[0, 10], title="CartView", unit="mK", format="%.2g", return_projected_map=True)

In [ ]:
plt.imshow(np.flipud(test_map), cmap="RdBu_r")
plt.colorbar()
plt.show()

In [ ]:
# answer from https://stackoverflow.com/questions/57619206/how-to-turn-healpy-region-into-2d-array
# alternate method

# Build a map
nside = 1024
npix = hp.nside2npix(nside)
# hpxmap = np.arange(npix)

# Get the cutout via a cartesian projection
lonra = [0, 10]
latra = [0, 10]

print(np.degrees(hp.nside2resol(nside)))

proj = hp.projector.CartesianProj(
    lonra=lonra, latra=latra,
    coord='G',
    xsize=256, ysize=256)
# reproj_im = proj.projmap(ngmap[0], vec2pix_func=partial(hp.vec2pix, nside))

# # Plot the cutout
# plt.imshow(reproj_im, origin='lower', interpolation='nearest')
# plt.show()

In [ ]:
# Saving 10 maps from a single realization as a beginning step
# This program has an issue. A new plot is generated each time cartview is run. How to turn off?
# def cutSqPatches(fullsky_map, npix_side, side_deg, num_patches=10):
#     """Cuts square patches by default (Even number for now)... without taking into account edge cases... with a simple method
    
#     Output: numpy array with (num_patches, npix_side, npix_side) shape
#     """
    
#     # Initialize numpy array
#     Tmap_datat = np.zeros((num_patches//2, int(npix_side), int(npix_side)))
#     Tmap_datab = np.zeros((num_patches//2, int(npix_side), int(npix_side)))

#     for counter in range(num_patches//2):
#         Tmap_datat[counter] = np.ma.getdata(hp.cartview(ngmap[0], fig=0, xsize=256, ysize=256, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[0, 10],
#                                                        title="CartView", unit="mK", format="%.2g", return_projected_map=True))
#         Tmap_datab[counter] = np.ma.getdata(hp.cartview(ngmap[0], fig=0, xsize=256, ysize=256, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[-10, 0],
#                                                        title="CartView", unit="mK", format="%.2g", return_projected_map=True))
#     return np.concatenate((Tmap_datat, Tmap_datab))


In [ ]:
# flatsky_patches = cutSqPatches(ngmap[0], xnpix, 10)
# print(flatsky_patches.shape)

# plt.imshow(flatsky_patches[5], cmap="RdBu_r") # remember images are flipped
# plt.colorbar()
# plt.show()

# Manipulating all realizations (Producing initial data set of 10000)

In [ ]:
# Defining relevant functions

def ngTemperatureMapFromFits(alml_fname, almnl_fname, f_NL_val, pol=True):
    lhdus = (1, 2, 3) if pol else 1
    alm_l = hp.read_alm(alml_fname, hdu=lhdus)
    alm_nl = hp.read_alm(almnl_fname, hdu=lhdus)
    alm = alm_l + (f_NL * alm_nl)
    
    ngmap = hp.alm2map(alm, 512, pol=pol)
    return ngmap[0]

def cutSqPatches(fullsky_map, npix_side, side_deg, num_patches=10):
    """Cuts square patches by default (Even number for now)... without taking into account edge cases... with a simple method
    
    Output: numpy array with (num_patches, npix_side, npix_side) shape
    """
    # Prevents a bug with cartview 
    import pylab
    wasinteractive = pylab.isinteractive()
    pylab.ioff()
    
    # Initialize numpy array
    Tmap_datat = np.zeros((num_patches//2, int(npix_side), int(npix_side)))
    Tmap_datab = np.zeros((num_patches//2, int(npix_side), int(npix_side)))

    for counter in range(num_patches//2):
        Tmap_datat[counter] = np.ma.getdata(hp.cartview(fullsky_map, xsize=256, ysize=256, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[0, 10],
                                                       title="CartView", unit="mK", format="%.2g", return_projected_map=True))
        Tmap_datab[counter] = np.ma.getdata(hp.cartview(fullsky_map, xsize=256, ysize=256, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[-10, 0],
                                                       title="CartView", unit="mK", format="%.2g", return_projected_map=True))
    if wasinteractive:
        pylab.ion()
        pylab.draw()
        
    return np.concatenate((Tmap_datat, Tmap_datab))

In [ ]:
# Creating all file names ahead of time
num_files = 1000
data_path = "/users/stevensonb/scratch/mlpng-fits/alm_"
fnames = [(data_path + "l_" + str(num).zfill(4) + "_v3.fits", data_path + "nl_" + str(num).zfill(4) + "_v3.fits") for num in range(1, num_files+1)]

In [ ]:
%%time
from IPython.display import clear_output
# Creating the initial data set of 10000 maps
# res_arcmin = 2.34375
xnpix = ynpix = 256
# m_deg_size = int(xnpix * (res_arcmin / 60))
m_deg_size = 10 # in cart projection
f_NL_base = 100

Tmap_data = np.zeros((10000, xnpix, ynpix))
npatches_per_map = patch_set = 10
f_NL_labels = np.zeros((10000, 1))
for count, f in enumerate(fnames):
    clear_output(wait=True)
    
    if np.random.random() < 0.9:
        f_NL = f_NL_base
    else:
        f_NL = 0

    full_Tmap = ngTemperatureMapFromFits(f[0], f[1], f_NL)
    Tmap_data[(0 + patch_set*count) : patch_set*(count+1)] = cutSqPatches(full_Tmap, xnpix, m_deg_size, npatches_per_map)
    print((0 + patch_set*count), "-" ,patch_set*(count+1), "with f_NL of", f_NL)

    # saving f_NL values used
    f_NL_labels[(0 + patch_set*count) : patch_set*(count+1)] = f_NL

In [ ]:
np.savez("NGTmap_100_fNL", Tmapdata=Tmap_data, fNLs=f_NL_labels)

In [ ]:
%%time
from IPython.display import clear_output
# Creating the initial data set of 10000 maps
# res_arcmin = 2.34375
xnpix = ynpix = 256
# m_deg_size = int(xnpix * (res_arcmin / 60))
m_deg_size = 10 # in cart projection
f_NL_base = 100

#Tmap_data = np.zeros((10000, xnpix, ynpix))
Tmap_data = np.zeros((100, xnpix, ynpix))
npatches_per_map = patch_set = 10
f_NL_labels = np.zeros((100, 1))
for count, f in enumerate(fnames):
    clear_output(wait=True)
    
    # Setting different values of f_NL every 100 mauniformmormormormorm    if count % 10 == 0:
    f_NL_base = int(np.random.uniform(50, 151))
    
    if np.random.random() < 0.9:
        f_NL = f_NL_base
    else:
        f_NL = 0

    full_Tmap = ngTemperatureMapFromFits(f[0], f[1], f_NL)
    Tmap_data[(0 + patch_set*count) : patch_set*(count+1)] = cutSqPatches(full_Tmap, xnpix, m_deg_size, npatches_per_map)
    print((0 + patch_set*count), "-" , patch_set*(count+1), "with", "f_NL of", f_NL)

    # saving f_NL values used
    f_NL_labels[(0 + patch_set*count) : patch_set*(count+1)] = f_NL
    if count >= 4:
        break

In [ ]:
np.savez("NGTmap_sampled_fNL_TEST", Tmapdata=Tmap_data, fNLs=f_NL_labels)